In [1]:

from google.colab import drive
drive.mount('/content/drive/')

# Go to your project folder
%cd "/content/drive/My Drive/Colab Notebooks/AI_in_BE_Python_Labs/"

import os
print("Working directory:", os.getcwd())

import numpy as np
import pandas as pd
import random

# For scaling
from sklearn.preprocessing import MinMaxScaler

# For table display in the demo
from IPython.display import display

# Set seeds for reproducibility (mainly for baselines)
np.random.seed(42)
random.seed(42)

Mounted at /content/drive/
/content/drive/My Drive/Colab Notebooks/AI_in_BE_Python_Labs
Working directory: /content/drive/My Drive/Colab Notebooks/AI_in_BE_Python_Labs


In [2]:
# === 2. LOAD COUNTY-LEVEL QUALITY-OF-LIFE DATA ===

# Make sure this file exists in: AI_in_BE_Python_Labs/data/
df = pd.read_csv("data/qol_county_level.csv")

print("Data shape:", df.shape)
df.head()

Data shape: (3134, 34)


,countyhelper,LSTATE,NMCNTY,FIPS,LZIP,ULOCALE,Overall Rank,2022 Population,2016 Crime Rate,Unemployment,...,1p3c,1p4c,2p0c,2p1c,2p2c,2p3c,2p4c,Stu:Tea Rank,Diversity Rank (Race),Diversity Rank (Gender)
0,VACharles City County,VA,Charles City County,51036,23030,42-Rural: Distant,NaN,"6,605",8/1000,3.21%,...,111.16%,119.90%,67.97%,90.07%,105.57%,123.82%,131.87%,135,1,25
1,TXMcmullen County,TX,McMullen County,48311,78072,43-Rural: Remote,NaN,576,47/1000,1.81%,...,105.46%,111.95%,72.03%,90.73%,104.21%,120.05%,127.11%,3,2,87
2,TXTerrell County,TX,Terrell County,48443,79848,43-Rural: Remote,NaN,693,20/1000,3.54%,...,127.10%,135.84%,87.96%,110.73%,125.11%,145.91%,153.79%,12,3,47
3,AKSkagway Municipality,AK,Skagway Municipality,2230,99840,43-Rural: Remote,NaN,"1,081",13/1000,7.19%,...,121.39%,128.32%,69.18%,94.01%,113.02%,132.18%,139.30%,15,4,9
4,GABaker County,GA,Baker County,13007,39870,42-Rural: Distant,NaN,"2,788",0,4.19%,...,122.57%,131.91%,85.54%,108.73%,124.45%,141.99%,153.63%,26,5,60


In [3]:
# === 3. SELECT FEATURES FOR THE RECOMMENDER ===

# These are the numeric QoL dimensions we will use
feature_cols = [
    'Cost of Living',
    '2022 Median Income',
    'AQI%Good',
    'ParkScore2023 Rank',
    '%CvgCityPark',
    'Unemployment',
    'Diversity Rank (Race)',
    'Diversity Rank (Gender)'
]

# Check that all columns exist
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns in df: {missing}")

In [4]:
# === 4. CLEAN FEATURES (REMOVE COMMAS/%, CONVERT TO NUMERIC) ===

subset = df[feature_cols].copy()

for col in subset.columns:
    # Remove commas and percent signs if present, then convert to numeric
    subset[col] = (
        subset[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('%', '', regex=False)
    )
    subset[col] = pd.to_numeric(subset[col], errors='coerce')

# Fill any remaining missing values with column medians
subset = subset.fillna(subset.median(numeric_only=True))

print("NaNs after cleaning:", subset.isna().sum())
subset.head()

NaNs after cleaning: Cost of Living             3134
2022 Median Income         3134
AQI%Good                      0
ParkScore2023 Rank            0
%CvgCityPark                  0
Unemployment                  0
Diversity Rank (Race)         0
Diversity Rank (Gender)       0
dtype: int64


,Cost of Living,2022 Median Income,AQI%Good,ParkScore2023 Rank,%CvgCityPark,Unemployment,Diversity Rank (Race),Diversity Rank (Gender)
0,NaN,NaN,93.76,-1,-1.0,3.21,1,25
1,NaN,NaN,75.33,-1,-1.0,1.81,2,87
2,NaN,NaN,75.33,-1,-1.0,3.54,3,47
3,NaN,NaN,87.86,-1,-1.0,7.19,4,9
4,NaN,NaN,83.30,-1,-1.0,4.19,5,60


In [5]:
# === 5. NORMALIZE FEATURES TO 0–1 RANGE ===

scaler = MinMaxScaler()
X = scaler.fit_transform(subset)

# Keep as DataFrame for easier debugging & alignment
X_df = pd.DataFrame(X, columns=feature_cols, index=df.index)

print("Normalized feature matrix shape:", X_df.shape)
X_df.head()

Normalized feature matrix shape: (3134, 8)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


,Cost of Living,2022 Median Income,AQI%Good,ParkScore2023 Rank,%CvgCityPark,Unemployment,Diversity Rank (Race),Diversity Rank (Gender)
0,NaN,NaN,0.819736,0.0,0.0,0.231446,0.000000,0.007660
1,NaN,NaN,0.225411,0.0,0.0,0.154480,0.000319,0.027450
2,NaN,NaN,0.225411,0.0,0.0,0.249588,0.000638,0.014682
3,NaN,NaN,0.629474,0.0,0.0,0.450247,0.000958,0.002553
4,NaN,NaN,0.482425,0.0,0.0,0.285322,0.001277,0.018832


In [6]:
# === 6. IDENTIFIER COLUMNS FOR DISPLAY ===

id_cols = []
for col in ['NMCNTY', 'LSTATE', 'County', 'State']:
    if col in df.columns:
        id_cols.append(col)

print("ID columns used:", id_cols)

ID columns used: ['NMCNTY', 'LSTATE']


In [7]:
# === 7. BASELINE 1: UNWEIGHTED SCORING (ALL FACTORS = 1) ===

baseline_weights = np.ones(len(feature_cols))
baseline_scores = X_df.values @ baseline_weights

baseline_df = df[id_cols].copy()
baseline_df['Score'] = baseline_scores

baseline_top = baseline_df.sort_values('Score', ascending=False).head(10)
print("Top 10 locations (unweighted baseline):")
baseline_top

Top 10 locations (unweighted baseline):


,NMCNTY,LSTATE,Score
0,Charles City County,VA,NaN
1,McMullen County,TX,NaN
2,Terrell County,TX,NaN
3,Skagway Municipality,AK,NaN
4,Baker County,GA,NaN
5,Geary County,KS,NaN
6,Hinsdale County,CO,NaN
7,Cochran County,TX,NaN
8,Armstrong County,TX,NaN
9,Delta County,TX,NaN


In [8]:
# === 8. BASELINE 2: RANDOM SCORES (SANITY CHECK) ===

random_scores = np.random.rand(len(df))

random_df = df[id_cols].copy()
random_df['Score'] = random_scores

random_top = random_df.sort_values('Score', ascending=False).head(10)
print("Top 10 locations (random baseline):")
random_top

Top 10 locations (random baseline):


,NMCNTY,LSTATE,Score
531,Bandera County,TX,0.999718
1464,Benton County,MO,0.999414
1954,Isabella County,MI,0.998348
1054,Orleans County,NY,0.997934
2529,Dougherty County,GA,0.997821
1209,Lincoln County,KY,0.997125
847,Putnam County,IL,0.996874
1727,Baxter County,AR,0.996697
532,Ste. Genevieve County,MO,0.996637
1912,DeWitt County,TX,0.996334


In [9]:
# === 9. RULE-BASED "PSEUDO-LLM" PREFERENCE EXTRACTOR ===
# Converts user text into feature importance weights in [0, 1].

import re

# Keywords associated with each factor
keyword_map = {
    'Cost of Living': [
        'cheap', 'affordable', 'low cost', 'low rent', 'budget', 'expensive', 'cost of living'
    ],
    '2022 Median Income': [
        'high income', 'good jobs', 'well paying', 'salary', 'wages', 'income'
    ],
    'AQI%Good': [
        'clean air', 'air quality', 'pollution', 'smog'
    ],
    'ParkScore2023 Rank': [
        'parks', 'park access', 'park score'
    ],
    '%CvgCityPark': [
        'greenery', 'green space', 'trees', 'nature', 'parks'
    ],
    'Unemployment': [
        'unemployment', 'jobless', 'job rate'
    ],
    'Diversity Rank (Race)': [
        'diverse', 'diversity', 'multicultural', 'race'
    ],
    'Diversity Rank (Gender)': [
        'gender equality', 'gender diverse', 'inclusive', 'lgbtq', 'gender'
    ],
}

# Phrases that loosely indicate "I don't care about X"
downplay_patterns = [
    r"don't care", r"do not care", r"not important",
    r"doesn['’]t matter", r"does not matter"
]

def get_user_weights_from_text_rule_based(user_text: str) -> dict:
    """
    Convert a natural-language description of preferences into a dict of
    importance weights for each factor in feature_cols.

    This is a simple keyword-based "pseudo-LLM" that approximates how
    an LLM would infer preferences, without any external model.
    """
    text = user_text.lower()
    weights = {}

    for factor in feature_cols:
        # Start at neutral importance
        base = 0.5

        # Boost importance if factor-related keywords are mentioned
        hits = 0
        for kw in keyword_map.get(factor, []):
            if kw in text:
                hits += 1

        if hits >= 2:
            base = 0.9  # very important
        elif hits == 1:
            base = 0.7  # somewhat important

        # If user says something like "I don't care about cost", downweight
        for pat in downplay_patterns:
            if re.search(pat, text) and factor.split()[0].lower() in text:
                base = 0.2

        weights[factor] = base

    return weights

In [10]:
# === 10. SCORING FUNCTION ===

def score_locations_from_weights(weight_dict: dict) -> pd.Series:
    """
    Given a dict of feature -> weight, compute a weighted score for each row
    in X_df (the normalized feature matrix).
    """
    user_vec = np.array([weight_dict[col] for col in feature_cols])
    scores = X_df.values @ user_vec
    return pd.Series(scores, index=X_df.index, name="PreferenceScore")

In [11]:
# === 11. RECOMMENDER: TEXT -> WEIGHTS -> SCORES -> TOP-N ===

def recommend_locations_from_text(user_text: str, top_n: int = 5):
    """
    Full pipeline for the recommender:
    1. Parse user text into weights (pseudo-LLM).
    2. Score all locations with a weighted dot product.
    3. Return the top-N locations and the weights used.
    """
    weights = get_user_weights_from_text_rule_based(user_text)
    scores = score_locations_from_weights(weights)

    out_df = df[id_cols].copy()
    out_df['PreferenceScore'] = scores

    top = out_df.sort_values('PreferenceScore', ascending=False).head(top_n).copy()
    return top, weights

In [12]:
test_text = """
I want a very affordable place with lots of greenery, good air quality, and a diverse community.
Income isn't my top priority as long as the cost of living is low.
I don't care much about unemployment or gender diversity.
"""

top_recs, weights_used = recommend_locations_from_text(test_text, top_n=5)
print("Weights used:")
print(weights_used)
top_recs


Weights used:
{'Cost of Living': 0.2, '2022 Median Income': 0.7, 'AQI%Good': 0.7, 'ParkScore2023 Rank': 0.5, '%CvgCityPark': 0.7, 'Unemployment': 0.2, 'Diversity Rank (Race)': 0.2, 'Diversity Rank (Gender)': 0.2}


,NMCNTY,LSTATE,PreferenceScore
0,Charles City County,VA,NaN
1,McMullen County,TX,NaN
2,Terrell County,TX,NaN
3,Skagway Municipality,AK,NaN
4,Baker County,GA,NaN


In [13]:
# === 12. EXPLANATION UTILITIES ===

# Friendly labels for more natural phrasing
friendly_factor_names = {
    'Cost of Living': 'affordability and overall cost of living',
    '2022 Median Income': 'median income and earning potential',
    'AQI%Good': 'air quality and pollution levels',
    'ParkScore2023 Rank': 'access to parks and recreation',
    '%CvgCityPark': 'green space and park coverage',
    'Unemployment': 'job market stability and unemployment',
    'Diversity Rank (Race)': 'racial and cultural diversity',
    'Diversity Rank (Gender)': 'gender representation and inclusivity',
}

def build_place_name(row: pd.Series) -> str:
    """
    Build a human-readable place name from available ID columns.
    """
    parts = []
    for col in ['NMCNTY', 'County']:
        if col in row.index and pd.notna(row[col]):
            parts.append(str(row[col]))
    for col in ['LSTATE', 'State']:
        if col in row.index and pd.notna(row[col]):
            parts.append(str(row[col]))
    return ", ".join(parts) if parts else "this county"


def make_explanation_for_row(row: pd.Series, user_text: str, weights: dict) -> str:
    """
    Generate a short, natural explanation (printed outside the DataFrame)
    for why this location is a good match, using the top 2–3 weighted factors.
    """
    place_name = build_place_name(row)

    # Sort factors by weight (descending)
    sorted_factors = sorted(weights.items(), key=lambda kv: kv[1], reverse=True)
    top_factors = [f for f, w in sorted_factors[:3]]
    friendly_top = [friendly_factor_names.get(f, f) for f in top_factors]

    # First sentence: connect place to preferences
    sent1 = (
        f"{place_name} looks like a strong match for the preferences you described. "
        f"In your description, you emphasized things like {friendly_top[0]}"
    )
    if len(friendly_top) > 1:
        sent1 += f" and {friendly_top[1]}"
    sent1 += ", and this area scores well on those dimensions in the dataset."

    # Second sentence: mention one more strength or overall fit
    if len(friendly_top) > 2:
        sent2 = (
            f" It also performs relatively well in terms of {friendly_top[2]}, "
            f"which adds to its appeal for you."
        )
    else:
        sent2 = " Overall, its quality-of-life indicators line up closely with what you said you’re looking for."

    return sent1 + sent2

In [14]:
# === 13. INTERACTIVE DEMO ===

def run_demo(top_n: int = 5):
    """
    Interactive demo for MoveMatch:
    - Asks the user to describe their ideal place to move.
    - Infers preference weights from the text (rule-based).
    - Shows top-N recommended locations in a table.
    - Prints narrative explanations for each recommendation.
    """
    print("=== MoveMatch Demo ===")
    print("Describe your ideal place to move (or type 'quit' to exit).")
    user_text = input("Your description: ")

    if user_text.strip().lower() == "quit":
        print("Demo ended.")
        return

    # 1. Get recommendations based on the text
    top_recs, weights_used = recommend_locations_from_text(user_text, top_n=top_n)

    # 2. Show inferred weights
    print("\n--- Inferred Preference Weights (0 = low, 1 = high) ---")
    for k, v in weights_used.items():
        print(f"{k}: {v:.2f}")

    # 3. Show top-N as a table (quick overview)
    print(f"\n--- Top {top_n} Recommended Locations (table view) ---")
    display(top_recs)

    # 4. Print narrative explanations for each recommendation
    print(f"\n--- Explanations for Each Recommendation ---\n")
    for rank, (_, row) in enumerate(top_recs.iterrows(), start=1):
        place_name = build_place_name(row)
        explanation = make_explanation_for_row(row, user_text, weights_used)

        print(f"{rank}. {place_name}")
        print(explanation)
        print()  # blank line between recommendations

In [15]:
run_demo(top_n=5)

=== MoveMatch Demo ===
Describe your ideal place to move (or type 'quit' to exit).
Your description: “I want a location close to parks with lots of young people, racial diversity, and good schools.”

--- Inferred Preference Weights (0 = low, 1 = high) ---
Cost of Living: 0.50
2022 Median Income: 0.50
AQI%Good: 0.50
ParkScore2023 Rank: 0.70
%CvgCityPark: 0.70
Unemployment: 0.50
Diversity Rank (Race): 0.70
Diversity Rank (Gender): 0.50

--- Top 5 Recommended Locations (table view) ---


,NMCNTY,LSTATE,PreferenceScore
0,Charles City County,VA,NaN
1,McMullen County,TX,NaN
2,Terrell County,TX,NaN
3,Skagway Municipality,AK,NaN
4,Baker County,GA,NaN



--- Explanations for Each Recommendation ---

1. Charles City County, VA
Charles City County, VA looks like a strong match for the preferences you described. In your description, you emphasized things like access to parks and recreation and green space and park coverage, and this area scores well on those dimensions in the dataset. It also performs relatively well in terms of racial and cultural diversity, which adds to its appeal for you.

2. McMullen County, TX
McMullen County, TX looks like a strong match for the preferences you described. In your description, you emphasized things like access to parks and recreation and green space and park coverage, and this area scores well on those dimensions in the dataset. It also performs relatively well in terms of racial and cultural diversity, which adds to its appeal for you.

3. Terrell County, TX
Terrell County, TX looks like a strong match for the preferences you described. In your description, you emphasized things like access to par